# Databricks notebook source
##### Gold Layer Data Load: Daily General Ledger & P&L Summary (`gold_fact_daily_gl_summary`)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

###### 1. Initialize Gold Schema & Target Table Name

In [0]:
catalog_name = "glrecon"
gold_table_name = f"{catalog_name}.gold.gold_fact_daily_gl_summary"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.gold")

###### 2. Read Silver Transaction & Overhead Tables

In [0]:
df_stock = spark.table(f"{catalog_name}.silver.silver_stock_transactions")
df_cash = spark.table(f"{catalog_name}.silver.silver_cash_transactions")
df_overhead = spark.table(f"{catalog_name}.silver.silver_overhead_costs")

###### 3. Aggregate Daily Stock, Cash, and Overhead Metrics

In [0]:
daily_stock = (
    df_stock.groupBy(F.col("TransactionDate").cast("date").alias("SummaryDate"))
    .agg(
        F.sum(
            F.when(F.col("TransDescription") == "PURCHASE", F.col("Quantity") * F.col("UnitPrice"))
            .otherwise(0)
        ).cast("decimal(14,2)").alias("TotalStockPurchases"),
        F.sum(
            F.when(F.col("TransDescription") == "SALE", F.col("Quantity") * F.col("UnitPrice"))
            .otherwise(0)
        ).cast("decimal(14,2)").alias("TotalStockSales")
    )
)

In [0]:
daily_cash = (
    df_cash.groupBy(F.col("TransactionDate").cast("date").alias("SummaryDate"))
    .agg(
        F.sum(
            F.when(F.col("TransDescription") == "CUSTOMER RECEIPT", F.col("CashTransAmount"))
            .otherwise(0)
        ).cast("decimal(14,2)").alias("TotalCashInflows"),
        F.sum(
            F.when(F.col("TransDescription") == "SUPPLIER PAYMENT", F.col("CashTransAmount"))
            .otherwise(0)
        ).cast("decimal(14,2)").alias("TotalCashOutflows")
    )
)


In [0]:
daily_ovh = (
    df_overhead.groupBy(F.col("CostDate").cast("date").alias("SummaryDate"))
    .agg(
        F.sum("Amount").cast("decimal(14,2)").alias("TotalOverheadCosts")
    )
)
display(daily_cash)

In [0]:
df_daily_staged = (
    daily_stock.join(daily_cash, "SummaryDate", "full_outer")
    .join(daily_ovh, "SummaryDate", "full_outer")
    .na.fill(0)
    .withColumn(
        "GrossTradingMargin",
        (F.col("TotalStockSales") - F.col("TotalStockPurchases")).cast("decimal(14,2)")
    )
    .withColumn(
        "NetOperatingCashFlow",
        (F.col("TotalCashInflows") - F.col("TotalCashOutflows")).cast("decimal(14,2)")
    )
    .withColumn(
        "NetDailyProfitLoss",
        (F.col("GrossTradingMargin") - F.col("TotalOverheadCosts")).cast("decimal(14,2)")
    )
    .withColumn(
        "DailyUnreconciledVariance",
        (
            (F.col("TotalStockSales") - F.col("TotalCashInflows")) +
            (F.col("TotalStockPurchases") - F.col("TotalCashOutflows"))
        ).cast("decimal(14,2)")
    )
    .withColumn("GoldProcessedDate", F.current_timestamp())
)


In [0]:
display(df_daily_staged)

###### 4. Upsert into Gold Daily GL Summary Fact Table (`MERGE`)

In [0]:
if not spark.catalog.tableExists(gold_table_name):
    print(f"Table {gold_table_name} does not exist. Initializing Gold summary table...")
    
    df_daily_staged.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
    print("Gold daily GL summary table successfully created and populated.")

else:
    print(f"Table {gold_table_name} exists. Proceeding with MERGE pipeline...")
    
    gold_table = DeltaTable.forName(spark, gold_table_name)

    (
        gold_table.alias("target")
        .merge(
            df_daily_staged.alias("source"),
            "target.SummaryDate = source.SummaryDate"
        )
        .whenMatchedUpdate(
            condition="""
                NOT (target.TotalStockPurchases <=> source.TotalStockPurchases) OR
                NOT (target.TotalStockSales <=> source.TotalStockSales) OR
                NOT (target.TotalCashInflows <=> source.TotalCashInflows) OR
                NOT (target.TotalCashOutflows <=> source.TotalCashOutflows) OR
                NOT (target.TotalOverheadCosts <=> source.TotalOverheadCosts)
            """,
            set={
                "TotalStockPurchases": "source.TotalStockPurchases",
                "TotalStockSales": "source.TotalStockSales",
                "GrossTradingMargin": "source.GrossTradingMargin",
                "TotalCashInflows": "source.TotalCashInflows",
                "TotalCashOutflows": "source.TotalCashOutflows",
                "NetOperatingCashFlow": "source.NetOperatingCashFlow",
                "TotalOverheadCosts": "source.TotalOverheadCosts",
                "NetDailyProfitLoss": "source.NetDailyProfitLoss",
                "DailyUnreconciledVariance": "source.DailyUnreconciledVariance",
                "GoldProcessedDate": F.current_timestamp()
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Merge execution completed successfully.")



##### Read data from gl summary

In [0]:
%sql

SELECT * FROM glrecon.gold.gold_fact_daily_gl_summary ORDER BY SummaryDate